In [1]:
# Install required libraries
%pip install -q langchain wikipedia python-docx requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import textwrap
import requests
import wikipedia
from docx import Document

# Define your LM Studio Endpoint
# Ensure your local server is running at this address
LM_STUDIO_URL = "http://127.0.0.1:1234/v1/chat/completions"

def lm_studio_chat(prompt, history=None, endpoint=LM_STUDIO_URL):
    messages = []
    
    # Load history if it exists
    if history:
        for user_msg, ai_msg in history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": ai_msg})
    
    # Add the current prompt
    messages.append({"role": "user", "content": prompt})
    
    payload = {
        "messages": messages,
        "model": "qwen/qwen3-4b", # You can also just use "local-model"
        "stream": False,
        "temperature": 0.7
    }
    
    try:
        response = requests.post(endpoint, json=payload, timeout=120)
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]
    except Exception as e:
        return f"[LM Studio Error] Something went wrong: {e}"

# Set a custom User-Agent so Wikipedia doesn't block the request
wikipedia.set_user_agent("AgenticChatbot/1.0 (local_test@example.com)")

def wiki_search(query):
    query = query.strip()
    if not query:
        return "No query provided."
    try:
        # Step 1: Search for the best matching page title first
        search_results = wikipedia.search(query, results=1)
        
        if not search_results:
            return "No matching Wikipedia page found."
            
        best_match = search_results[0]
        print(f"[System] Wikipedia found best match: '{best_match}'")
        
        # Step 2: Fetch the summary of that exact page
        return wikipedia.summary(best_match, sentences=10)
        
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Multiple results found. Try being more specific: {e.options[:5]}"
    except wikipedia.exceptions.PageError:
        return "No page found for that query."
    except Exception as e:
        return f"Wiki search error: {e}"

def create_docx(text, filename="output.docx"):
    doc = Document()
    doc.add_paragraph(text)
    doc.save(filename)
    return filename

def print_wrapped(text, width=100):
    wrapper = textwrap.TextWrapper(width=width, replace_whitespace=False)
    lines = str(text).split('\n')
    wrapped_text = [wrapper.fill(line) for line in lines]
    print("\n".join(wrapped_text))

print("[System] Helpers initialized and ready.")

[System] Helpers initialized and ready.


In [3]:
# Main chatbot loop

conversation_history = []
conversation_stats = {
    "user_turns": 0,
    "wiki_searches": 0,
    "arithmetic": 0,
    "document_saves": 0,
    "ai_chats": 0
}

print("Agentic AI Chatbot initialized! Type 'q' to quit.\n")

while True:
    q = input("You: ")
    if q.strip().lower() == "q":
        print("\n--- Conversation Statistics ---")
        for k, v in conversation_stats.items():
            print(f"{k}: {v}")
        print("Goodbye!")
        break
        
    print(f"\n[User] \n{q}")
    conversation_stats["user_turns"] += 1
    
    # 1. Ask LM Studio to decide which tool to use
    print("\n[System] Deciding which tool to use...")
    decision_prompt = (
        f"Analyze this user input: '{q}'. "
        "Decide which tool to use from this exact list: [chat, wiki, arithmetic, save]. "
        "Use 'save' if the user asks to save, export, or create a document/docx. "
        "Reply STRICTLY in this format: TOOL: <tool> | KEYWORDS: <keywords>. "
        "Do not add any greetings or extra text."
    )
    
    lm_decision = lm_studio_chat(decision_prompt)
    print(f"[Tool Decision] {lm_decision}")
    
    # Parse the decision safely
    tool_match = re.search(r'TOOL:\s*(\w+)', lm_decision, re.IGNORECASE)
    tool = tool_match.group(1).lower() if tool_match else "chat"
    
    final_answer = ""
    
    # 2. Execute the chosen tool
    if tool == "wiki":
        conversation_stats["wiki_searches"] += 1
        keywords_match = re.search(r'KEYWORDS:\s*(.+)', lm_decision, re.IGNORECASE)
        keywords = keywords_match.group(1).strip() if keywords_match else q
        
        print(f"\n[System] Searching Wikipedia for: {keywords}...")
        wiki_text = wiki_search(keywords)
        
        final_answer = lm_studio_chat(f"Summarize this info for the user concisely: {wiki_text}")
        print("\n[AI (Wiki)]")
        print_wrapped(final_answer)

    elif tool == "arithmetic":
        conversation_stats["arithmetic"] += 1
        print("\n[System] Calculating...")
        final_answer = lm_studio_chat(f"Calculate and evaluate this expression step-by-step: {q}")
        print("\n[AI (Arithmetic)]")
        print_wrapped(final_answer)
        
    elif tool == "save":
        conversation_stats["document_saves"] += 1
        print("\n[System] Saving document...")
        
        # Check if there is a previous answer to save
        if conversation_history:
            last_user_msg, last_ai_msg = conversation_history[-1]
            
            # Try to name the file based on the keywords
            keywords_match = re.search(r'KEYWORDS:\s*(.+)', lm_decision, re.IGNORECASE)
            keywords = keywords_match.group(1).strip() if keywords_match else "output"
            safe_keyword = re.sub(r'[^A-Za-z0-9_-]+', '_', keywords).strip('_') or 'saved_output'
            doc_filename = f"{safe_keyword}.docx"
            
            create_docx(last_ai_msg, filename=doc_filename)
            final_answer = f"I successfully saved my last response as '{doc_filename}' in your folder!"
        else:
            final_answer = "There is nothing in our conversation history to save yet."
            
        print("\n[AI (Save)]")
        print_wrapped(final_answer)

    else:
        # Default Chat
        conversation_stats["ai_chats"] += 1
        print("\n[System] Thinking...")
        final_answer = lm_studio_chat(q, history=conversation_history)
        print("\n[AI (Chat)]")
        print_wrapped(final_answer)
    
    # 3. Update memory
    conversation_history.append((q, final_answer))

Agentic AI Chatbot initialized! Type 'q' to quit.


[User] 
Hi, I'm Suzette!!

[System] Deciding which tool to use...
[Tool Decision] 

TOOL: chat | KEYWORDS: Hi Suzette

[System] Thinking...

[AI (Chat)]


Hi there, Suzette! 😊 It's a pleasure to meet you! How are you doing? What have you been up to
lately? I’d love to hear more about you—whether it’s something fun you’ve been doing, a project
you’re working on, or just a random question! Feel free to share anything you want. 🌟

[User] 
doing great!! btw, I'm currently taking BS Computer Engineering any suggestion best career path?

[System] Deciding which tool to use...
[Tool Decision] 

TOOL: chat | KEYWORDS: BS Computer Engineering, career path, suggestion

[System] Thinking...

[AI (Chat)]


That’s awesome to hear! 🎉 Computer Engineering is a fantastic field with so many exciting
opportunities. Since you’re still in school, it’s a great time to explore different areas and figure
out what you're passionate about. Here are some caree